# Imports

In [23]:
from pathlib import Path
import os
import kagglehub
from collections import Counter
import shutil
from sklearn.model_selection import train_test_split

# Directory Setup Setup

In [10]:
os.environ["KAGGLEHUB_CACHE"] = "../kagglehub_cache"

# Downloading the dataset

In [11]:
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
print("Path to dataset files:", path)

100%|██████████| 2.29G/2.29G [02:09<00:00, 19.0MB/s] 

Extracting files...


Path to dataset files: ../kagglehub_cache/datasets/paultimothymooney/chest-xray-pneumonia/versions/2


# Merging the dataset images

In [13]:
base_dir = Path("../kagglehub_cache/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray")
splits = ["train", "test", "val"]
classes = ["NORMAL", "PNEUMONIA"]

print("Dataset root:", base_dir)

Dataset root: ../kagglehub_cache/datasets/paultimothymooney/chest-xray-pneumonia/versions/2/chest_xray


In [16]:
all_images = []
all_labels = []

for split in splits:
    for cls in classes:
        folder = base_dir / split / cls
        if not folder.exists():
            print(f"Warning: {folder} not found — skipping")
            continue
        
        images = list(folder.glob("*.jpeg"))  # or "*.jpg", "*.png" if mixed
        all_images.extend(images)
        all_labels.extend([cls] * len(images))

print(f"Total images collected: {len(all_images)}")
print("Class distribution:", Counter(all_labels))

Total images collected: 5856
Class distribution: Counter({'PNEUMONIA': 4273, 'NORMAL': 1583})


In [ ]:
# for checking correct merging
assert len(all_images) == len(all_labels), "Mismatch in paths/labels!"

# Splitting the dataset to 70/20/10 using Stratified Split

In [20]:
X_train, X_temp, y_train, y_temp = train_test_split(
    all_images,
    all_labels,
    test_size=0.10,          # 10% → final test
    stratify=all_labels,     # keep class balance
    random_state=42
)

In [21]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=1/3,           # of the 30% remaining → 10% overall test
    stratify=y_temp,
    random_state=42
)

print(f"Train: {len(X_train)} ({len(X_train)/len(all_images):.1%})")
print(f"Val:   {len(X_val)}   ({len(X_val)/len(all_images):.1%})")
print(f"Test:  {len(X_test)}  ({len(X_test)/len(all_images):.1%})")

print("Train class dist:", Counter(y_train))
print("Val   class dist:", Counter(y_val))
print("Test  class dist:", Counter(y_test))

Train: 5270 (90.0%)
Val:   390   (6.7%)
Test:  196  (3.3%)
Train class dist: Counter({'PNEUMONIA': 3845, 'NORMAL': 1425})
Val   class dist: Counter({'PNEUMONIA': 285, 'NORMAL': 105})
Test  class dist: Counter({'PNEUMONIA': 143, 'NORMAL': 53})


# Saving to datasets for further ML models

In [ ]:
new_root = Path("../datasets")
new_root.mkdir(exist_ok=True)

for name, paths, labels in [
    ("train", X_train, y_train),
    ("val",   X_val,   y_val),
    ("test",  X_test,  y_test)
]:
    split_dir = new_root / name
    split_dir.mkdir(exist_ok=True)
    
    for cls in classes:
        (split_dir / cls).mkdir(exist_ok=True)
    
    for img_path, label in zip(paths, labels):
        dest = split_dir / label / img_path.name
        shutil.copy2(img_path, dest) 

print("New split created at:", new_root)
print("Done! images are saved to ../datasets/{train,val,test}")

New split created at: ../datasets
Done! You can now point ImageFolder or tf.keras.utils.image_dataset_from_directory to ./my_custom_split_chest_xray/{train,val,test}
